# 05 Evaluation Report — matching, fusion и graph quality

Эта тетрадка объединяет бывшие `05_clustering_resolution.ipynb` и `06_evaluation_report.ipynb`. Она ничего не пересчитывает заново: читает результаты `03_matching_comparison.ipynb` и `04_fusion_pack_grouping.ipynb`, затем показывает итоговый benchmark, выбранный fusion-run, качество family/pack-графа и примеры ошибок.


## Как читать метрики

`precision` показывает, какая доля предсказанных связей действительно правильная. `recall` показывает, какую долю настоящих связей мы нашли. Для family false link опаснее, потому что он может склеить разные товары в один компонент; false split обычно дешевле, потому что дубль можно обработать позже.


## Блок кода 1. Подготовка окружения


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "research" / "dedup").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


## Блок кода 2. Загрузка готовых артефактов


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

LABELING_PATH = DATA_DIR / "labeling_sauces.csv"
SUMMARY_PATH = REPORTS_DIR / "binary_threshold_summary.csv"
PREDICTIONS_PATH = REPORTS_DIR / "binary_threshold_predictions.csv"
COMPONENTS_PATH = DATA_DIR / "fusion_components_sauces.csv"
PAIR_EVAL_PATH = DATA_DIR / "fusion_pair_eval_sauces.csv"
EVAL_SPLIT = os.environ.get("DEDUP_FUSION_EVAL_SPLIT", "test")

required_paths = [SUMMARY_PATH, PREDICTIONS_PATH, COMPONENTS_PATH, PAIR_EVAL_PATH]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required artifacts. Run notebooks/03_matching_comparison.ipynb "
        "and notebooks/04_fusion_pack_grouping.ipynb first: "
        + ", ".join(str(path) for path in missing)
    )

labels = pd.read_csv(LABELING_PATH) if LABELING_PATH.exists() else pd.DataFrame()
threshold_summary = pd.read_csv(SUMMARY_PATH)
threshold_predictions = pd.read_csv(PREDICTIONS_PATH)
components = pd.read_csv(COMPONENTS_PATH)
pair_eval = pd.read_csv(PAIR_EVAL_PATH)

print(f"Labels: {len(labels)} rows")
print(f"Threshold summary: {len(threshold_summary)} rows")
print(f"Threshold predictions: {len(threshold_predictions)} rows")
print(f"Fusion components: {len(components)} rows")
print(f"Fusion pair eval: {len(pair_eval)} rows")
if {"fusion_method", "fusion_threshold_strategy", "fusion_threshold_same"}.issubset(pair_eval.columns):
    display(pair_eval[["fusion_method", "fusion_threshold_strategy", "fusion_threshold_same"]].drop_duplicates())


## Блок кода 3. Состояние gold-set


In [ ]:
if labels.empty or "label" not in labels.columns:
    print("labeling_sauces.csv не найден или в нём нет label; пропускаем разбор разметки")
else:
    label_counts = labels["label"].fillna("<empty>").astype(str).value_counts(dropna=False).rename_axis("label").reset_index(name="pairs")
    display(label_counts)


## Блок кода 4. Сравнение methods на test

`test` здесь только для чтения результата. Выбор threshold уже был сделан на `dev` в `03`.


In [ ]:
test_summary = threshold_summary[threshold_summary["split"].astype(str).eq("test")].copy()
if test_summary.empty:
    test_summary = threshold_summary.copy()

ranking_cols = [
    "method",
    "threshold_strategy",
    "threshold_same",
    "precision",
    "recall",
    "f1",
    "false_merge_count",
    "false_split_count",
    "cost",
    "weighted_f1",
    "weighted_total_cost",
    "weight_source",
]
ranking_cols = [column for column in ranking_cols if column in test_summary.columns]
sort_cols = [column for column in ["weighted_total_cost", "cost", "false_merge_count", "false_split_count", "f1"] if column in test_summary.columns]
ascending = [True, True, True, True, False][: len(sort_cols)]
ranked = test_summary.sort_values(sort_cols, ascending=ascending) if sort_cols else test_summary

display(ranked[ranking_cols].head(20))


## Блок кода 5. Выбранный fusion-run и graph quality


In [ ]:
def _as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str) -> dict[str, object]:
    true_link = _as_bool(frame[true_col])
    pred_link = _as_bool(frame[pred_col])
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "scope": scope,
        "eval_split": frame["split"].iloc[0] if "split" in frame.columns and frame["split"].nunique() == 1 else "all",
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positive_links": tp,
        "false_links": fp,
        "missed_links": fn,
        "true_negative_links": tn,
    }


eval_pairs = pair_eval[pair_eval["split"].astype(str).eq(EVAL_SPLIT)].copy() if "split" in pair_eval.columns else pair_eval.copy()
if eval_pairs.empty:
    eval_pairs = pair_eval.copy()

link_report = pd.DataFrame(
    [
        _binary_link_report(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family"),
        _binary_link_report(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack"),
    ]
)
display(link_report)


## Блок кода 6. Размеры компонентов

Здесь важно быстро заметить огромные компоненты: они часто означают chaining из-за false merge.


In [ ]:
def _component_summary(frame: pd.DataFrame, component_col: str, graph: str) -> dict[str, object]:
    if component_col not in frame.columns or frame.empty:
        return {"graph": graph, "components": 0, "multi_node_components": 0, "max_nodes": 0}
    sizes = frame.groupby(component_col, dropna=True).size().rename("nodes").reset_index()
    return {
        "graph": graph,
        "components": int(sizes[component_col].nunique()),
        "multi_node_components": int((sizes["nodes"] > 1).sum()),
        "max_nodes": int(sizes["nodes"].max()) if not sizes.empty else 0,
    }


component_report = pd.DataFrame(
    [
        _component_summary(components, "fusion_family_id", "pred_family"),
        _component_summary(components, "fusion_pack_id", "pred_pack"),
        _component_summary(components, "true_family_id", "true_family_partial"),
        _component_summary(components, "true_pack_id", "true_pack_partial"),
    ]
)
display(component_report)

for component_col in ["fusion_family_id", "fusion_pack_id"]:
    if component_col in components.columns:
        sizes = components.groupby(component_col, dropna=True).size().rename("nodes").reset_index()
        display(sizes.sort_values("nodes", ascending=False).head(15))


## Блок кода 7. Опасные false merge и ошибки графа


In [ ]:
selected_method = None
selected_strategy = None
if {"fusion_method", "fusion_threshold_strategy"}.issubset(pair_eval.columns) and not pair_eval.empty:
    selected_method = str(pair_eval["fusion_method"].dropna().iloc[0])
    selected_strategy = str(pair_eval["fusion_threshold_strategy"].dropna().iloc[0])

danger = threshold_predictions.copy()
if selected_method is not None and "method" in danger.columns:
    danger = danger[danger["method"].astype(str).eq(selected_method)]
if selected_strategy is not None and "threshold_strategy" in danger.columns:
    danger = danger[danger["threshold_strategy"].astype(str).eq(selected_strategy)]
if "false_merge" in danger.columns:
    danger = danger[_as_bool(danger["false_merge"])]
else:
    danger = danger.iloc[0:0]

danger_cols = [
    "split",
    "score",
    "threshold_same",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
    "pair_weight",
]
danger_cols = [column for column in danger_cols if column in danger.columns]
print("Selected threshold false merges")
display(danger[danger_cols].head(20))


def _show_graph_examples(frame: pd.DataFrame, *, true_col: str, pred_col: str, title: str) -> None:
    true_link = _as_bool(frame[true_col])
    pred_link = _as_bool(frame[pred_col])
    cols = [
        "split",
        "score",
        "same_base_product",
        "same_pack_signature",
        "title_a",
        "title_b",
        "brand_a",
        "brand_b",
        "unit_amount_a",
        "unit_amount_b",
        "total_amount_a",
        "total_amount_b",
        "multipack_count_a",
        "multipack_count_b",
    ]
    cols = [column for column in cols if column in frame.columns]
    print()
    print(f"{title}: false links")
    display(frame[~true_link & pred_link][cols].head(10))
    print()
    print(f"{title}: missed links")
    display(frame[true_link & ~pred_link][cols].head(10))


_show_graph_examples(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", title="Family")
_show_graph_examples(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", title="Pack")


## Блок кода 8. Итог человеческим языком


In [ ]:
conclusions = []
if not ranked.empty:
    best = ranked.iloc[0]
    strategy = best["threshold_strategy"] if "threshold_strategy" in best else "<unknown>"
    cost = best["cost"] if "cost" in best else "<unknown>"
    conclusions.append(
        f"По test-таблице верхний кандидат в текущей сортировке: {best['method']} / {strategy} с cost={cost}."
    )
if not link_report.empty:
    family = link_report[link_report["scope"].eq("family")].iloc[0]
    pack = link_report[link_report["scope"].eq("pack")].iloc[0]
    conclusions.append(
        f"Family graph: precision={family['precision']:.3f}, recall={family['recall']:.3f}, false_links={int(family['false_links'])}."
    )
    conclusions.append(
        f"Pack graph: precision={pack['precision']:.3f}, recall={pack['recall']:.3f}, false_links={int(pack['false_links'])}."
    )
conclusions.append(
    "Разные фасовки не являются отдельным ML-классом: они выделяются после matching через unit/total/multipack signature."
)
display(pd.DataFrame({"conclusion": conclusions}))


## Что делать после отчёта

Если family false links выглядят опасно, возвращаемся в `03` и выбираем более строгий method/strategy или улучшаем scorer/reranker на hard negatives. Если family уже достаточно чистая, можно отдельно улучшать pack signature и готовить перенос выбранного подхода из research в production pipeline.
